<div dir="rtl" style="text-align: right;">
<h3>1 کتابخانه‌ها و نمودار</h3>
</div>

In [ ]:
import os
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from dotenv import load_dotenv
from scipy import stats
from sqlalchemy import URL, create_engine, text, bindparam
from statsmodels.stats.proportion import proportions_ztest
import statsmodels.api as sm

pio.templates.default = 'plotly_dark'
pio.renderers.default = 'notebook'
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:.3f}'.format)

current_dir = Path.cwd().resolve()

ROOT = next(
    (
        path
        for path in [current_dir, *current_dir.parents]
        if (path / "notebooks" / "nba_analysis.ipynb").is_file()
    ),
    None,
)

if ROOT is None:
    raise FileNotFoundError(
        "Project root was not found. Run the notebook from within the repository."
    )

notebooks_dir = str(ROOT / "notebooks")

if notebooks_dir not in sys.path:
    sys.path.insert(0, notebooks_dir)

import nba_top_players
from partially_overlapping_tests import partially_overlapping_t

load_dotenv(ROOT / ".env")

DB_PATH = ROOT / "database" / "data" / "nba_star.db"

if not DB_PATH.is_file():
    raise FileNotFoundError(f"Database file was not found: {DB_PATH}")

<div dir="rtl" style="text-align: right;">
<h3>1.1 آزمون فرض</h3>
</div>

<div dir="rtl" style="text-align: right;">
<h2>2. خواندن داده‌ها</h2>
</div>

In [ ]:
engine = create_engine(
    URL.create("sqlite", database=str(DB_PATH))
)

with engine.connect() as connection:
    integrity_result = connection.execute(
        text("PRAGMA integrity_check")
    ).scalars().all()

    table_names = connection.execute(
        text(
            "SELECT name FROM sqlite_master "
            "WHERE type = 'table' ORDER BY name"
        )
    ).scalars().all()

if integrity_result != ["ok"]:
    raise RuntimeError(f"Database integrity check failed: {integrity_result}")

required_tables = {
    "fact_player_stats",
    "dim_player",
    "dim_season",
    "dim_team_scope",
    "dim_player_context",
    "dim_phase",
}

missing_tables = required_tables - set(table_names)

if missing_tables:
    raise RuntimeError(f"Required tables are missing: {sorted(missing_tables)}")

print(f"Database: {DB_PATH}")
print("Integrity check: OK")
print(f"Tables: {len(table_names)}")

<div dir="rtl" style="text-align: right;">
<h2>3. پاک‌سازی داده‌ها</h2>
<h3>3.1 داده های خالی</h3>

</div>

<div dir="rtl" style="text-align: right;">
<h3>3.2 داده‌های پرت</h3>
</div>

<div dir="rtl" style="text-align: right;">
<h3>3.3 بازیکنان واجد شرایط و فعال</h3>
</div>

<div dir="rtl" style="text-align: right;">
<h2>4. شناخت داده‌ها</h2>
<h3>4.1 ویژگی‌های بدنی</h3>
</div>

<div dir="rtl" style="text-align: right;">
<h3>4.2 عملکرد بازیکنان</h3>
</div>

<div dir="rtl" style="text-align: right;">
<h3>4.3 تیم‌ها و قهرمانان</h3>
</div>

<div dir="rtl" style="text-align: right;">
<h3>4.4 رابطه میان آمار بازیکنان</h3>
</div>

<div dir="rtl" style="text-align: right;">
<h3>4.5 مقایسه پست‌ها در پنج فصل</h3>
</div>

<div dir="rtl" style="text-align: right;">
<h2>5. معیار انتخاب بازیکنان برتر</h2>
</div>

<div dir="rtl" style="text-align: right;">

### هدف
برای هر فصل، بازیکنان را با یک **امتیاز واحد** رتبه‌بندی می‌کنیم و 50 نفر اول را انتخاب می‌کنیم.

### چرا یک امتیاز ترکیبی؟
هیچ آمار منفردی عملکرد را کامل نشان نمی‌دهد. برای مثال، مجموع امتیاز به تعداد شوت‌ها وابسته است و کیفیتِ بازی در هر دقیقه را نشان نمی‌دهد. به همین دلیل دو دیدگاه را با هم ترکیب کردیم:

- **شاخص‌های نرخی (کیفیت در هر دقیقه):**
  - **BPM** (Box Plus/Minus): تخمین امتیازی که بازیکن در هر 100 حمله نسبت به یک بازیکن متوسط اضافه می‌کند (صفر = متوسط).
  - **PER** (Player Efficiency Rating): نسبت کارهای مثبت (امتیاز، ریباند، پاس گل، سرقت، بلاک) به کارهای منفی (شوت از دست رفته، توپ‌باخت، خطا) در هر دقیقه؛ میانگین لیگ حدود 15 است.
  - **WS/48** (Win Shares per 48 minutes): سهم بازیکن در بردهای تیم به ازای 48 دقیقه؛ میانگین لیگ حدود 0.1 است.
- **شاخص حجمی (مجموع ارزش در کل فصل):**
  - **WS** (Win Shares): تعداد بردهایی که بازیکن در کل فصل برای تیمش ساخته است. این شاخص باعث می‌شود بازیکنی که نیمی از فصل را غایب بوده، فقط به خاطر نرخ بالا در صدر نیاید.

### مراحل محاسبه
1. فقط **فصل عادی** و **یک ردیف برای هر بازیکن**؛ برای بازیکنانی که وسط فصل تیم عوض کرده‌اند، ردیف مجموعِ همهٔ تیم‌ها (2TM/3TM) استفاده می‌شود.
2. فقط بازیکنان با **حداقل 1000 دقیقه** بازی؛ زیرا آمار نمونه‌های کوچک پرنوسان است و بازیکن با چند دقیقه بازیِ عالی نباید قهرمان شود.
3. **استانداردسازی (z-score)** هر شاخص در همان فصل: 
   $z(x)=\dfrac{x-\mu}{\sigma}$ 
   که $\mu$ و $\sigma$ میانگین و انحراف معیار همان شاخص بین بازیکنان واجد شرایط آن فصل است. این کار شاخص‌هایی با مقیاس‌های متفاوت را قابل جمع می‌کند.
4. محاسبهٔ امتیاز نهایی:

$$\text{Impact}=\frac{z(\text{BPM})+z(\text{PER})+z(\text{WS/48})}{3}$$

$$\text{Score}=0.6\times\text{Impact}+0.4\times z(\text{WS})$$

5. مرتب‌سازی نزولی بر اساس Score و انتخاب 50 نفر اول.

### چرا وزن‌های 0.6 و 0.4؟
کیفیت در هر دقیقه نشانهٔ اصلی توانایی است (وزن بیشتر)، ولی میزان حضور در فصل هم مهم است. این وزن‌ها یک **تصمیم قضاوتی** هستند، نه فرمول «درستِ مطلق»؛ به همین دلیل حساسیت‌سنجی انجام دادیم.


</div>


In [ ]:
top50 = nba_top_players.get_top_players(engine, "2019-20")        # or just pass the path :)))
# all_years = pd.concat([nba_top_players.get_top_players(engine, s) for s in
#                        ["2019-20","2020-21","2021-22","2022-23","2023-24"]])
top50

<div dir="rtl" style="text-align: right;">
<h2>6. آمار توصیفی</h2>
<h3>6.1 قد نامزدهای MVP و بازیکنان برتر</h3>
</div>

<div dir="rtl" style="text-align: right;">
<h3>6.2 تجربه و قد بازیکنان تیم قهرمان</h3>
</div>

<div dir="rtl" style="text-align: right;">
<h3>6.3 پیشنهاد پوینت‌گارد</h3>
</div>

<div dir="rtl" style="text-align: right;">
<h2>7. آزمون فرض</h2>
<h3>7.1 تغییر چابکی بازیکنان برتر</h3>
</div>

<div dir="rtl" style="direction: rtl; text-align: right; line-height: 2;">

<h4 dir="rtl" style="direction: rtl; text-align: right;">
تعریف مسئله
</h4>

<p dir="rtl" style="direction: rtl; text-align: right;">
در این بخش بررسی می‌کنیم آیا میانگین شاخص چابکیِ ۲۰ بازیکن برتر هر فصل، در دورهٔ جدید نسبت به دورهٔ قدیم افزایش یافته است یا خیر. مطابق تعریف صورت پروژه، شاخص چابکی از تقسیم قد بازیکن بر وزن او محاسبه می‌شود.
</p>

<h4 dir="rtl" style="direction: rtl; text-align: right;">
دوره‌های مقایسه
</h4>

<p dir="rtl" style="direction: rtl; text-align: right;">
<strong>دورهٔ قدیم:</strong>
فصل‌های
<span dir="ltr" style="direction: ltr; unicode-bidi: isolate; display: inline-block; white-space: nowrap;">2020-21</span>
و
<span dir="ltr" style="direction: ltr; unicode-bidi: isolate; display: inline-block; white-space: nowrap;">2021-22</span>.
</p>

<p dir="rtl" style="direction: rtl; text-align: right;">
<strong>دورهٔ جدید:</strong>
فصل‌های
<span dir="ltr" style="direction: ltr; unicode-bidi: isolate; display: inline-block; white-space: nowrap;">2022-23</span>
و
<span dir="ltr" style="direction: ltr; unicode-bidi: isolate; display: inline-block; white-space: nowrap;">2023-24</span>.
</p>

<h4 dir="rtl" style="direction: rtl; text-align: right;">
معیار انتخاب بازیکنان
</h4>

<p dir="rtl" style="direction: rtl; text-align: right;">
برای یکسان‌بودن معیار انتخاب در بخش‌های مختلف پروژه، از تابع مشترک تیم با نام
<span dir="ltr" style="direction: ltr; unicode-bidi: isolate; display: inline-block; white-space: nowrap;"><code>get_top_players</code></span>
استفاده می‌کنیم. برای هر فصل، ۲۰ بازیکن نخست بر اساس امتیاز ترکیبی این تابع انتخاب می‌شوند. این تابع، آمار فصل عادی و حداقل ۱۰۰۰ دقیقه بازی را مبنا قرار می‌دهد و برای بازیکنان چندتیمی از آمار تجمیعی فصل استفاده می‌کند.
</p>

<h4 dir="rtl" style="direction: rtl; text-align: right;">
تعریف شاخص و واحد مشاهده
</h4>

<p dir="rtl" style="direction: rtl; text-align: right;">
شاخص چابکی را با استفاده از واحدهای موجود در دیتابیس،
از تقسیم قد برحسب اینچ بر وزن برحسب پوند محاسبه می‌کنیم.
واحدها برای تمام بازیکنان و فصل‌ها یکسان نگه داشته می‌شوند.
سپس میانگین نسبت‌های فردی را در دو دوره مقایسه می‌کنیم.
</p>

<p dir="rtl" style="direction: rtl; text-align: right;">
واحد مشاهده، «بازیکن در یک فصل» است. بنابراین انتظار داریم در هر فصل ۲۰ رکورد و در هر دوره ۴۰ رکورد داشته باشیم.
</p>

<h4 dir="rtl" style="direction: rtl; text-align: right;">
فرضیه‌های آماری
</h4>

<p dir="rtl" style="direction: rtl; text-align: right;">
<strong>فرض صفر:</strong>
میانگین شاخص در دورهٔ جدید، کمتر یا مساوی دورهٔ قدیم است.
</p>

<p dir="rtl" style="direction: rtl; text-align: right;">
<strong>فرض مقابل:</strong>
میانگین شاخص در دورهٔ جدید، بیشتر از دورهٔ قدیم است.
</p>

<p dir="rtl" style="direction: rtl; text-align: right;">
سطح معناداری را پیش از اجرای آزمون برابر با ۰٫۰۵ در نظر می‌گیریم.
</p>

</div>

In [ ]:
top_2020_21 = nba_top_players.get_top_players(db=engine, season="2020-21", n=20)

top_2021_22 = nba_top_players.get_top_players(db=engine, season="2021-22", n=20)

top_2022_23 = nba_top_players.get_top_players(db=engine, season="2022-23", n=20)

top_2023_24 = nba_top_players.get_top_players(db=engine, season="2023-24", n=20)

all_top_players = pd.concat(
    [top_2020_21, top_2021_22, top_2022_23, top_2023_24],
    ignore_index=True,
)

print(f"Shape of DF : {all_top_players.shape}\n",
      f"Value Counts based on season : \n{all_top_players.value_counts(subset="season")}\n",
      f"Duplicate counts based on season and player key:{all_top_players.duplicated(subset=["player_key", "season"]).sum()}\n")

In [ ]:
top_query = text("""
    SELECT player_key, listed_height_inches, listed_weight_pounds
    FROM dim_player
    WHERE player_key IN :player_keys
""")
top_query = top_query.bindparams(
    bindparam("player_keys", expanding=True)
)

with engine.connect() as connection:
    player_measurements = pd.read_sql_query(
        top_query,
        connection,
        params={"player_keys": all_top_players["player_key"].unique().tolist()},
    )

print(
    f"{len(all_top_players["player_key"].unique())}\n",
    f"{len(player_measurements)}\n",
    f"{player_measurements.isna().sum()}\n",
    f"{player_measurements.duplicated(subset="player_key").sum()}\n"
)

In [ ]:
merged_top_df = pd.merge(
    all_top_players,
    player_measurements,
    on="player_key",
    how="left",
    validate="many_to_one",
)

print(
    f"{merged_top_df.shape}\n",
    f"{merged_top_df.isna().sum()}\n",
    f"{merged_top_df.duplicated(subset=["player_key", "season"]).sum()}\n"
    f"{((merged_top_df["listed_height_inches"] > 0) & (merged_top_df["listed_weight_pounds"] > 0)).all()}\n"
)

merged_top_df["agility_ratio"] = merged_top_df["listed_height_inches"] / merged_top_df["listed_weight_pounds"]

season_map = {
    "2020-21": "old",
    "2021-22": "old",
    "2022-23": "new",
    "2023-24": "new",
}

merged_top_df["period"] = merged_top_df["season"].map(season_map)

In [ ]:
period_summary = merged_top_df.groupby("period")["agility_ratio"].describe()

season_summary = merged_top_df.groupby("season")["agility_ratio"].describe()

display(period_summary)
display(season_summary)

<div dir="rtl" style="direction: rtl; text-align: right; line-height: 2;">

<h4 dir="rtl" style="direction: rtl; text-align: right;">
برداشت اولیه از آمار توصیفی
</h4>

<p dir="rtl" style="direction: rtl; text-align: right;">
هر دوره شامل ۴۰ رکورد بازیکن‌ـ‌فصل است.
میانگین شاخص قد به وزن در دورهٔ جدید حدود ۰٫۳۵۲
و در دورهٔ قدیم حدود ۰٫۳۴۵ است.
میانه نیز از حدود ۰٫۳۳۹ در دورهٔ قدیم
به ۰٫۳۴۶ در دورهٔ جدید افزایش یافته است.
بنابراین، هر دو معیار مرکزی در دورهٔ جدید مقدار بیشتری دارند.
</p>

<p dir="rtl" style="direction: rtl; text-align: right;">
بررسی جداگانهٔ فصل‌ها نشان می‌دهد میانگین شاخص
در سه فصل نخست افزایش یافته، اما در فصل آخر کاهش پیدا کرده است.
در نتیجه، بالاتربودن میانگین دورهٔ جدید
به معنای افزایش پیوستهٔ شاخص در تمام فصل‌ها نیست.
</p>

<p dir="rtl" style="direction: rtl; text-align: right;">
این نتایج، توصیف تفاوت مشاهده‌شده در داده‌ها هستند
و به‌تنهایی برای نتیجه‌گیری دربارهٔ معناداری آماری
یا تعمیم آن به فصل‌های دیگر کافی نیستند.
اعداد این بخش به‌صورت تقریبی گزارش شده‌اند.
</p>

</div>

In [ ]:
player_season_mean_difference = (
    period_summary.loc["new", "mean"]
    - period_summary.loc["old", "mean"]
)

player_season_percentage_change = (
    player_season_mean_difference
    / period_summary.loc["old", "mean"]
) * 100

print(
    player_season_mean_difference,
    player_season_percentage_change,
)

In [ ]:
fig_box = px.box(merged_top_df, x="period", y="agility_ratio")
fig_box.update_traces(boxmean=True)
fig_box.show()

In [ ]:
player_season_count_distribution = merged_top_df.groupby("player_key")["season"].nunique().value_counts()

unique_players_per_period = merged_top_df.groupby("period")["player_key"].nunique()

player_period_count_distribution = merged_top_df.groupby("player_key")["period"].nunique().value_counts()

display(
    player_season_count_distribution
    .rename_axis("season_count")
    .to_frame(name="player_count")
)

display(
    unique_players_per_period
    .to_frame(name="unique_players")
)

display(
    player_period_count_distribution
    .rename_axis("period_count")
    .to_frame(name="player_count")
)

<div dir="rtl" style="direction: rtl; text-align: right; line-height: 2;">

<h4 dir="rtl" style="direction: rtl; text-align: right;">
ساختار مشاهدات و آماده‌سازی برای آزمون
</h4>

<p dir="rtl" style="direction: rtl; text-align: right;">
۸۰ مشاهدهٔ بازیکن‌ـ‌فصل، متعلق به ۴۱ بازیکن متفاوت است.
دورهٔ قدیم ۲۹ بازیکن متفاوت و دورهٔ جدید ۲۶ بازیکن متفاوت دارد.
از این میان، ۱۴ بازیکن در هر دو دوره حضور دارند.
بنابراین، دو گروه نه کاملاً مستقل و نه کاملاً زوجی هستند.
</p>

<p dir="rtl" style="direction: rtl; text-align: right;">
برای بررسی تکمیلی با آزمون نمونه‌های دارای هم‌پوشانی جزئی،
برای هر بازیکن در هر دوره فقط یک مقدار در نظر می‌گیریم.
این مقدار از میانگین شاخص او در فصل‌های منتخب همان دوره به دست می‌آید.
در این تحلیل، هر بازیکن در هر دوره وزن برابر دارد؛
در حالی که در تحلیل توصیفی قبلی، هر حضور فصلی یک سهم داشت.
</p>

<p dir="rtl" style="direction: rtl; text-align: right;">
قد و وزن ثبت‌شده برای هر بازیکن در دیتابیس ثابت است.
بنابراین، مقایسهٔ حاضر تفاوت ترکیب بازیکنان منتخب را بررسی می‌کند
و نشان‌دهندهٔ تغییر بدنی همان بازیکنان در طول زمان نیست.
</p>

</div>

In [ ]:
player_period = (
    merged_top_df
    .groupby(["player_key", "period"])["agility_ratio"]
    .mean()
    .unstack("period")
    .reindex(columns=["old", "new"])
)

paired = player_period.dropna(subset=["old", "new"])

new_only = player_period.loc[
    player_period["old"].isna(), "new"
].to_numpy()

old_only = player_period.loc[
    player_period["new"].isna(), "old"
].to_numpy()

paired_new = paired["new"].to_numpy()
paired_old = paired["old"].to_numpy()

distinct_player_summary = player_period.describe().T

distinct_player_mean_difference = distinct_player_summary.loc["new", "mean"] - distinct_player_summary.loc["old", "mean"]
distinct_player_percentage_change = (distinct_player_mean_difference / distinct_player_summary.loc["old", "mean"]) * 100

display(distinct_player_summary)

print(
    f"Mean difference: {distinct_player_mean_difference:.6f}\n"
    f"Percentage change: {distinct_player_percentage_change:.2f}%"
)

In [ ]:
old_values = player_period["old"].dropna().to_numpy()
new_values = player_period["new"].dropna().to_numpy()

# Use identical bin boundaries for both periods.
all_values = np.concatenate([old_values, new_values])
bin_edges = np.histogram_bin_edges(all_values, bins=15)

fig, axes = plt.subplots(
    2, 2,
    figsize=(12, 8),
    sharex="row",
    sharey="row",
)

period_data = [
    ("Old", old_values),
    ("New", new_values),
]

for column, (period_name, values) in enumerate(period_data):
    # Each observation contributes an equal percentage within its period.
    weights = np.full(len(values), 100.0 / len(values))

    axes[0, column].hist(
        values,
        bins=bin_edges,
        weights=weights,
        color="royalblue",
        edgecolor="white",
        alpha=0.85,
        rwidth=0.95,
    )

    axes[0, column].set_title(
        f"{period_name} period — Histogram (n={len(values)})"
    )
    axes[0, column].set_xlabel("Agility ratio (in/lb)")
    axes[0, column].set_ylabel("Percent")
    axes[0, column].grid(axis="y", alpha=0.2)
    axes[0, column].set_axisbelow(True)

    sm.qqplot(
        values,
        fit=True,
        line="45",
        ax=axes[1, column],
    )

    axes[1, column].set_title(
        f"{period_name} period — Normal Q-Q plot"
    )
    axes[1, column].set_xlabel("Theoretical normal quantiles")
    axes[1, column].set_ylabel("Standardized sample quantiles")
    axes[1, column].grid(alpha=0.2)
    axes[1, column].set_axisbelow(True)

fig.suptitle(
    "Agility ratio distribution — Distinct players",
    fontsize=14,
)

plt.tight_layout()
plt.show()

In [ ]:
shapiro_old_w, shapiro_old_p = stats.shapiro(
    old_values
)

shapiro_new_w, shapiro_new_p = stats.shapiro(
    new_values
)

normality_summary = pd.DataFrame({
    "period": ["old", "new"],
    "n": [
        player_period["old"].count(),
        player_period["new"].count(),
    ],
    "W": [shapiro_old_w, shapiro_new_w],
    "p_value": [shapiro_old_p, shapiro_new_p],
}).set_index("period")

display(normality_summary)

<div dir="rtl" style="direction: rtl; text-align: right; line-height: 2;">

<h4 dir="rtl" style="direction: rtl; text-align: right;">
بررسی شکل توزیع شاخص
</h4>

<p dir="rtl" style="direction: rtl; text-align: right;">
بررسی توزیع در این بخش بر اساس بازیکنان متفاوت هر دوره انجام شد؛
یعنی ۲۹ مقدار برای دورهٔ قدیم و ۲۶ مقدار برای دورهٔ جدید.
هیستوگرام‌ها تمرکز اصلی مقادیر دو دوره را در محدوده‌های نزدیک
و دنبالهٔ راست بلندتر دورهٔ قدیم را نشان می‌دهند.
</p>

<p dir="rtl" style="direction: rtl; text-align: right;">
در نمودارهای چندک‌ـ‌چندک، بخش زیادی از نقاط خط مرجع نرمال را
تقریباً دنبال می‌کنند؛ بااین‌حال، در مقادیر بالای دورهٔ قدیم
فاصلهٔ بیشتری از خط دیده می‌شود.
بنابراین، سازگاری با توزیع نرمال کامل نیست
و انحراف انتهای بالایی باید در تفسیر لحاظ شود.
</p>

<p dir="rtl" style="direction: rtl; text-align: right;">
در آزمون شاپیروـ‌ویلک، فرض صفر نرمال‌بودن توزیع است.
مقدار احتمال برای دورهٔ قدیم حدود ۰٫۱۰۸
و برای دورهٔ جدید حدود ۰٫۷۱۲ به دست آمد.
هر دو مقدار از سطح معناداری ۰٫۰۵ بیشترند؛
بنابراین، شواهد کافی برای رد نرمال‌بودن هیچ‌یک از دو گروه نداریم.
این نتیجه به معنی اثبات نرمال‌بودن نیست،
به‌ویژه با توجه به تعداد محدود مشاهدات.
</p>

<p dir="rtl" style="direction: rtl; text-align: right;">
نتایج نموداری و آزمون نرمال‌بودن در کنار هم تفسیر می‌شوند.
این بررسی‌ها به‌تنهایی استقلال بازیکنان،
تصادفی‌بودن سازوکار انتخاب یا تمام فرض‌های آزمون مقاله را تأیید نمی‌کنند.
هیچ مشاهده‌ای صرفاً به دلیل فاصله از خط مرجع یا دورافتاده‌بودن حذف نشد.
</p>

</div>

<div dir="rtl" style="direction: rtl; text-align: right; line-height: 2;">

<h4 dir="rtl" style="direction: rtl; text-align: right;">
روش آزمون اختلاف میانگین
</h4>

<p dir="rtl" style="direction: rtl; text-align: right;">
پس از درنظرگرفتن یک مقدار برای هر بازیکن در هر دوره،
دو گروه شامل ۱۴ بازیکن مشترک، ۱۲ بازیکن فقط در دورهٔ جدید
و ۱۵ بازیکن فقط در دورهٔ قدیم هستند.
بنابراین، آزمون مستقل معمولی وابستگی افراد مشترک را نادیده می‌گیرد
و آزمون زوجی معمولی نیز افراد غیرمشترک را کنار می‌گذارد.
</p>

<p dir="rtl" style="direction: rtl; text-align: right;">
برای تحلیل تکمیلی، از آزمون نمونه‌های دارای هم‌پوشانی جزئی،
نسخهٔ
<span dir="ltr" style="direction: ltr; unicode-bidi: isolate; display: inline-block; white-space: nowrap;">T<sub>new2</sub></span>
استفاده می‌کنیم.
این روش برابری واریانس دو گروه را فرض نمی‌کند
و همبستگی مشاهدات زوجی را در برآورد خطای استاندارد اختلاف لحاظ می‌کند.
میانگین و انحراف معیار هر گروه از تمام بازیکنان همان گروه
و همبستگی از بازیکنان مشترک با تطبیق شناسه محاسبه می‌شود.
</p>

<p dir="rtl" style="direction: rtl; text-align: right;">
پارامتر مورد بررسی، میانگین دورهٔ جدید منهای میانگین دورهٔ قدیم،
با وزن برابر برای هر بازیکن در هر دوره است.
فرض صفر، کمتر یا مساوی صفر بودن این اختلاف
و فرض مقابل، مثبت بودن آن است.
بنابراین، آزمون یک‌طرفهٔ راست با سطح معناداری ۰٫۰۵ اجرا می‌شود.
این آزمون مستقیماً میانگین وزن‌داده‌شده بر اساس تعداد حضورهای فصلی را نمی‌آزماید.
</p>

<p dir="rtl" style="direction: rtl; text-align: right;">
به دلیل انتخاب غیرتصادفی بازیکنان، تفاوت سازوکار انتخاب با
گم‌شدن کاملاً تصادفی داده و همبستگی دقیقاً یک در جفت‌های مشترک،
انطباق کامل با شرایط بررسی‌شده در مقاله تأیید نشده است.
بنابراین، آزمون به‌عنوان تحلیل اکتشافی و مشروط به فرض‌های روش گزارش می‌شود.
</p>

<h4 dir="rtl" style="direction: rtl; text-align: right;">
منبع روش
</h4>

<p dir="ltr" style="direction: ltr; text-align: left; line-height: 1.8;">
Derrick, B., White, P., &amp; Toher, D. (2019).
Parametric and Non-Parametric Tests for the Comparison of Two Samples
Which Both Include Paired and Unpaired Observations.
Journal of Modern Applied Statistical Methods, 18(1), eP2847.
<a href="https://doi.org/10.22237/jmasm/1556669520">
https://doi.org/10.22237/jmasm/1556669520
</a>
</p>

</div>

In [ ]:
paper_result = partially_overlapping_t(
    new_only=new_only,
    old_only=old_only,
    paired_new=paired_new,
    paired_old=paired_old,
)

display(pd.DataFrame([paper_result]).T.rename(columns={0: "value"}))

alpha = 0.05

if paper_result["p_value_one_sided"] < alpha:
    print("Reject H0 at alpha = 0.05.")
else:
    print("Fail to reject H0 at alpha = 0.05.")

<div dir="rtl" style="direction: rtl; text-align: right; line-height: 2;">

<h4 dir="rtl" style="direction: rtl; text-align: right;">
نتیجهٔ آزمون و جمع‌بندی
</h4>

<p dir="rtl" style="direction: rtl; text-align: right;">
در تحلیل توصیفی فهرست‌های فصلی، میانگین نسبت قد به وزن
در دورهٔ جدید حدود ۲٫۰۲٪ بیشتر از دورهٔ قدیم بود.
در این محاسبه، هر حضور بازیکن در یک فصل یک مشاهده محسوب می‌شود.
</p>

<p dir="rtl" style="direction: rtl; text-align: right;">
برای تحلیل تکمیلی با آزمون نمونه‌های دارای هم‌پوشانی جزئی،
برای هر بازیکن در هر دوره یک مقدار در نظر گرفتیم.
در نتیجه، گروه قدیم شامل ۲۹ بازیکن و گروه جدید شامل ۲۶ بازیکن بود
که ۱۴ نفر از آن‌ها در هر دو گروه حضور داشتند.
با این وزن‌دهی، اختلاف میانگین جدید منهای قدیم حدود ۰٫۰۰۵۳۳۱
و افزایش نسبی حدود ۱٫۵۴٪ بود.
</p>

<p dir="rtl" style="direction: rtl; text-align: right;">
نسخهٔ آزمون بدون فرض برابری واریانس‌ها اجرا شد.
آمارهٔ آزمون حدود ۰٫۸۱۱، درجهٔ آزادی حدود ۳۲٫۵۰
و مقدار احتمال یک‌طرفه حدود ۰٫۲۱۲ به دست آمد.
از آنجا که این مقدار از سطح معناداری ۰٫۰۵ بیشتر است،
فرض صفر رد نمی‌شود.
بنابراین، تحت فرض‌های روش، شواهد آماری کافی برای
افزایش میانگین شاخص در تحلیل بازیکنان متفاوت به دست نیامد.
این نتیجه به معنی اثبات برابری میانگین‌ها یا نبود افزایش نیست.
</p>

<p dir="rtl" style="direction: rtl; text-align: right;">
فاصلهٔ اطمینان دوطرفهٔ ۹۵٪ اختلاف میانگین،
از حدود منفی ۰٫۰۰۸۰۵۵ تا مثبت ۰٫۰۱۸۷۱۸ بود.
کران پایین یک‌طرفهٔ ۹۵٪ نیز حدود منفی ۰٫۰۰۵۸۰۳ به دست آمد.
این نتایج، عدم‌قطعیت قابل‌توجه دربارهٔ جهت و اندازهٔ اختلاف را نشان می‌دهند.
</p>

<h4 dir="rtl" style="direction: rtl; text-align: right;">
محدودیت‌های تفسیر
</h4>

<p dir="rtl" style="direction: rtl; text-align: right;">
قد و وزن ثبت‌شده برای هر بازیکن در دیتابیس ثابت است؛
بنابراین، اختلاف مشاهده‌شده بازتاب تفاوت ترکیب بازیکنان منتخب
و وزن‌دهی به حضور آن‌هاست، نه تغییر بدنی همان افراد.
نسبت قد به وزن نیز شاخص تعریف‌شدهٔ پروژه است
و اندازه‌گیری مستقیم چابکی ورزشی محسوب نمی‌شود.
</p>

<p dir="rtl" style="direction: rtl; text-align: right;">
انتخاب بازیکنان تصادفی نیست و انتخاب‌نشدن یک فرد در یک دوره
را نمی‌توان گم‌شدن کاملاً تصادفی داده دانست.
همچنین، استقلال بازیکنان متفاوت فرض شده
و همبستگی بازیکنان مشترک به دلیل ثابت بودن مشخصات بدنی برابر یک است.
ازاین‌رو، انطباق کامل داده‌ها با شرایط بررسی‌شده در مقاله تأیید نشده
و نتیجهٔ آزمون به‌صورت اکتشافی و مشروط به فرض‌های روش گزارش می‌شود.
تعمیم نتیجه به همهٔ بازیکنان یا یک روند تاریخی عمومی،
با داده‌های چهار فصل حاضر توجیه نمی‌شود.
</p>

</div>

<div dir="rtl" style="text-align: right;">
<h3>7.2 تجربه بازیکنان تیم قهرمان</h3>
</div>

<div dir="rtl" style="text-align: right;">
<h2>8. تحلیل‌های تکمیلی</h2>

<h3>8.1 رابطه حقوق و عملکرد بازیکنان</h3>

### سؤال
آیا بازیکنانی که حقوق بیشتری می‌گیرند، واقعاً عملکرد بهتری دارند؟ «عملکرد» را با همان **امتیاز (Score)** بخش 5 می‌سنجیم؛ ولی این بار برای **همهٔ بازیکنان واجد شرایط** هر فصل (نه فقط 50 نفر برتر)، چون برای دیدن رابطه، هم بازیکنان کم‌درآمد و هم پردرآمد لازم‌اند.

### تصمیم‌های روش‌شناسی
- **لگاریتم حقوق:** توزیع حقوق به‌شدت به راست چوله است (چند ستارهٔ بسیار پردرآمد). لگاریتم آن را متقارن می‌کند و رابطه را «درصدی» می‌سازد (مثلاً دو برابر شدن حقوق).
- **مقایسه درون هر فصل:** سقف حقوق هر سال بالاتر می‌رود (میانگین حقوق از حدود 7.5 به 9.3 میلیون دلار)، پس حقوق را در همان فصل استاندارد (z-score) می‌کنیم.
- **همبستگی اسپیرمن:** به داده پرت حساس نیست و فقط رابطهٔ «هرچه حقوق بیشتر، امتیاز بیشتر» را می‌سنجد؛ برای حقوق‌های خیلی بالا مناسب‌تر از پیرسون است.
- **تفکیک قرارداد تازه‌کاران:** حقوق بازیکنان سال‌های اول (تجربهٔ 3 سال یا کمتر) تقریباً با جایگاه آن‌ها در درفت تعیین می‌شود، نه با عملکردشان؛ پس جداگانه بررسی‌شان می‌کنیم.

</div>

In [ ]:
import statsmodels.formula.api as smf
from sqlalchemy import MetaData, Table, select

SEASONS = ["2019-20", "2020-21", "2021-22", "2022-23", "2023-24"]
ROOKIE_SCALE_MAX_EXP = 3


def load_salary_performance(engine, seasons=SEASONS):
    """One row per qualified player-season: Score (section 5) + salary + experience."""
    meta = MetaData()
    stats_t, season_t, phase_t, ctx_t, sal_t = (
        Table(name, meta, autoload_with=engine)
        for name in ("fact_player_stats", "dim_season", "dim_phase",
                     "dim_player_context", "fact_player_salary")
    )

####performance score for ALL qualified players (huge n on purpose = no top-50 cut)
    scores = pd.concat(
        [nba_top_players.get_top_players(engine, s, n=10_000) for s in seasons],
        ignore_index=True,
    )

####salary per player-season
    q_salary = (
        select(sal_t.c.player_key,
               season_t.c.season_label.label("season"),
               sal_t.c.annual_salary_usd.label("salary"))
        .select_from(sal_t.join(season_t, sal_t.c.season_key == season_t.c.season_key))
    )

####years of experience (canonical regular-season row of each player-season)
    q_experience = (
        select(stats_t.c.player_key,
               season_t.c.season_label.label("season"),
               ctx_t.c.experience_years.label("experience"))
        .select_from(
            stats_t.join(season_t, stats_t.c.season_key == season_t.c.season_key)
                   .join(phase_t, stats_t.c.phase_key == phase_t.c.phase_key)
                   .join(ctx_t, stats_t.c.context_key == ctx_t.c.context_key)
        )
        .where(phase_t.c.phase_code == "regular_season",
               stats_t.c.is_player_phase_total == 1)
    )

    with engine.connect() as conn:
        salary = pd.read_sql(q_salary, conn)
        experience = pd.read_sql(q_experience, conn)

    df = (scores.merge(salary, on=["player_key", "season"], how="left")
                .merge(experience, on=["player_key", "season"], how="left"))

    n_missing = int(df["salary"].isna().sum())
    df = df.dropna(subset=["salary"]).copy()
    print(f"Qualified player-seasons: {len(df) + n_missing} | without salary (dropped): {n_missing}")
    print(f"Rows used: {len(df)} | distinct players: {df['player_key'].nunique()}")

    df["log_salary"] = np.log(df["salary"])
    df["salary_z"] = df.groupby("season")["log_salary"].transform(lambda x: (x - x.mean()) / x.std())
    df["salary_quartile"] = df.groupby("season")["salary"].transform(
        lambda x: pd.qcut(x, 4, labels=["Q1 (lowest)", "Q2", "Q3", "Q4 (highest)"])
    )
    df["contract_group"] = np.where(df["experience"] <= ROOKIE_SCALE_MAX_EXP,
                                    "Rookie scale (exp <= 3)", "Veteran (exp >= 4)")
    df["salary_musd"] = df["salary"] / 1e6
    return df


salary_df = load_salary_performance(engine)
salary_df[["season", "player_name", "team", "experience", "salary_musd", "score", "rank"]].head()

<div dir="rtl" style="text-align: right;">

<h4>توزیع حقوق: چرا لگاریتم؟</h4>

</div>

In [ ]:
print(f"Skewness of salary:      {salary_df['salary'].skew():.2f}")
print(f"Skewness of log(salary):  {salary_df['log_salary'].skew():.2f}")

fig = px.histogram(salary_df, x="salary_musd", nbins=40, title="Salary distribution (million USD)")
fig.show()
fig = px.histogram(salary_df, x="log_salary", nbins=40, title="Salary distribution (natural log of USD)")
fig.show()

<div dir="rtl" style="text-align: right;">

<h4>رابطهٔ کلی حقوق و امتیاز</h4>

نمودار زیر هر بازیکن-فصل را نشان می‌دهد (محور افقی لگاریتمی است). دو خط روند جدا برای تازه‌کاران و باتجربه‌ها رسم شده است.

</div>

In [ ]:
fig = px.scatter(
    salary_df, x="salary", y="score", color="contract_group",
    log_x=True, trendline="ols", trendline_options=dict(log_x=True),
    hover_data=["player_name", "season", "experience"], opacity=0.4,
    labels={"salary": "Salary (USD, log scale)", "score": "Performance score"},
    title="Salary vs performance (qualified players, 2019-20 to 2023-24)",
)
fig.show()

In [ ]:
def corr_row(g):
    rho, p = stats.spearmanr(g["log_salary"], g["score"])
    r, _ = stats.pearsonr(g["log_salary"], g["score"])
    return pd.Series({"n": len(g), "spearman_rho": rho, "p_value": p, "pearson_r_log": r})


corr_table = salary_df.groupby("season")[["log_salary", "score"]].apply(corr_row)
corr_table.loc["all seasons"] = corr_row(salary_df)
corr_table["n"] = corr_table["n"].astype(int)
corr_table["p_value"] = corr_table["p_value"].map("{:.1e}".format)
display(corr_table)

r2 = smf.ols("score ~ salary_z", salary_df).fit().rsquared
print(f"R-squared of score ~ salary (within-season z): {r2:.3f}")

<div dir="rtl" style="text-align: right;">

**نتیجه:** در هر 5 فصل همبستگی اسپیرمن مثبت و معنادار است (حدود 0.39 تا 0.48؛ روی کل داده‌ها 0.44). یعنی بازیکنان پردرآمدتر **معمولاً** امتیاز بالاتری دارند، ولی رابطه فقط متوسط است: حقوق حدود **21٪** از تغییرات امتیاز را توضیح می‌دهد (R²). 

یک بازیکن می‌تواند در چند فصل تکرار شود، پس ردیف‌ها مستقل نیستند و p-value ردیف «کل فصل‌ها» خوش‌بینانه است. برای همین ردیف‌های هر فصل را هم جدا گزارش کردیم.

</div>

<div dir="rtl" style="text-align: right;">

<h4>مقایسهٔ گروه‌های حقوقی</h4>

بازیکنان هر فصل را به 4 گروه حقوقی تقسیم می‌کنیم و میانگین امتیاز را مقایسه می‌کنیم. برای اینکه ببینیم تفاوت‌ها تصادفی نیستند از آزمون ناپارامتریک **کروسکال-والیس** و برای مقایسهٔ حقوق 50 نفر برتر با بقیه از **من-ویتنی** استفاده می‌کنیم.

</div>

In [ ]:
quartile_summary = salary_df.groupby("salary_quartile")["score"].agg(["count", "mean", "median"])
display(quartile_summary)

fig = px.bar(quartile_summary.reset_index(), x="salary_quartile", y="mean",
             labels={"salary_quartile": "Salary quartile (within season)", "mean": "Mean score"},
             title="Mean performance score by salary quartile")
fig.show()

h, p_kw = stats.kruskal(*[g["score"].values for _, g in salary_df.groupby("salary_quartile")])
print(f"Kruskal-Wallis across quartiles: H = {h:.1f}, p = {p_kw:.1e}")

is_top50 = salary_df["rank"] <= 50
u, p_mw = stats.mannwhitneyu(salary_df.loc[is_top50, "salary"],
                             salary_df.loc[~is_top50, "salary"], alternative="greater")
print(f"Median salary: top 50 = ${salary_df.loc[is_top50, 'salary'].median():,.0f} | "
      f"others = ${salary_df.loc[~is_top50, 'salary'].median():,.0f} (Mann-Whitney p = {p_mw:.1e})")

<div dir="rtl" style="text-align: right;">

**نتیجه:** میانگین امتیاز از گروه کم‌درآمدترین (Q1، حدود 0.42 منفی) تا پردرآمدترین (Q4، حدود 0.73 مثبت) یکنواخت بالا می‌رود. میانهٔ حقوق 50 نفر برتر هر فصل حدود **25.9 میلیون دلار** است، در برابر حدود **6.9 میلیون** برای بقیه؛ هر دو آزمون تفاوت را معنادار نشان می‌دهند.

</div>

<div dir="rtl" style="text-align: right;">

<h4>تازه‌کاران در برابر باتجربه‌ها</h4>

اگر رابطه برای همه یکسان نیست، نباید یک عدد برای همه گزارش کنیم. حقوق بازیکنان سال‌های اول عمدتاً با جایگاه درفت تعیین می‌شود. پس همبستگی را برای دو گروه جدا حساب می‌کنیم.

</div>

In [ ]:
group_corr = (salary_df.groupby(["contract_group", "season"])[["log_salary", "score"]]
              .apply(lambda g: stats.spearmanr(g["log_salary"], g["score"])[0])
              .unstack())
group_corr["all seasons"] = salary_df.groupby("contract_group")[["log_salary", "score"]].apply(
    lambda g: stats.spearmanr(g["log_salary"], g["score"])[0])
group_corr["R-squared"] = pd.Series({
    grp: smf.ols("score ~ salary_z", g).fit().rsquared
    for grp, g in salary_df.groupby("contract_group")})
group_corr["n"] = salary_df.groupby("contract_group").size()
display(group_corr)

<div dir="rtl" style="text-align: right;">

**نتیجه:** برای بازیکنان باتجربه همبستگی حدود **0.50** است (R² حدود 0.23) ولی برای تازه‌کاران فقط حدود **0.17** (R² حدود 0.04). یعنی در سال‌های اول، حقوق تقریباً هیچ اطلاعاتی دربارهٔ عملکرد نمی‌دهد؛ رابطهٔ کلی بخش قبل بیشتر از بازیکنان باتجربه می‌آید.

</div>

<div dir="rtl" style="text-align: right;">

<h4>چه کسانی بیشتر/کمتر از حقوقشان بازی می‌کنند؟</h4>

برای باتجربه‌ها (که حقوقشان به عملکرد مرتبط‌تر است) یک مدل خطی `score ~ salary_z + season` برازش می‌دهیم. **باقیمانده (residual)** مثبت یعنی بازیکن بهتر از چیزی بازی کرده که حقوقش پیش‌بینی می‌کند، و منفی یعنی بدتر.

</div>

In [ ]:
vets = salary_df[salary_df["experience"] > ROOKIE_SCALE_MAX_EXP].copy()
model = smf.ols("score ~ salary_z + C(season)", vets).fit()
vets["residual"] = model.resid
print(f"Veteran model: R-squared = {model.rsquared:.3f}, slope of salary_z = {model.params['salary_z']:.3f}")

cols = ["season", "player_name", "team", "salary_musd", "score", "residual"]
print("\nPlay far BETTER than their salary predicts:")
display(vets.nlargest(8, "residual")[cols])
print("Play far WORSE than their salary predicts:")
display(vets.nsmallest(8, "residual")[cols])

<div dir="rtl" style="text-align: right;">

<h4>جمع‌بندی حقوق و عملکرد</h4>

1. **بین حقوق و عملکرد رابطهٔ مثبت و معنادار ولی متوسط وجود دارد** (اسپیرمن 0.44، R² حدود 0.21). بازار حقوق عملکرد را تا حد خوبی منعکس می‌کند، اما فقط یک‌پنجم تغییرات را توضیح می‌دهد.
2. **رابطه برای همه یکسان نیست:** برای باتجربه‌ها قوی‌تر (0.50) و برای تازه‌کاران ضعیف (0.17) است؛ چون حقوق تازه‌کاران را عمدتاً جایگاه درفت تعیین می‌کند.
3. **ابرستاره‌ها (مثل یوکیچ و یانیس) بسیار بیشتر از حقوقشان ارزش می‌سازند** و در رأس فهرست «بهتر از حقوق» هستند. یک توضیح محتمل (که در این تحلیل آزموده نشده) سقف حداکثر حقوق قرارداد است. در طرف مقابل، بازیکنانی مثل جردن پول و جان وال کمتر از حقوقشان بازی کرده‌اند.

### محدودیت‌ها
- **همبستگی یعنی علّیت نیست:** قراردادها معمولاً **قبل از** فصل و بر اساس عملکرد گذشته بسته می‌شوند؛ پس حقوق «پاداش گذشته» است نه «پیش‌بینی آینده».
- امتیاز ما از آمار جعبه‌ای می‌آید و بخشی از ارزش بازیکن (مثل دفاع) را نمی‌بیند، پس «بدتر از حقوق» ممکن است تا حدی خطای معیار باشد.
- مصدومیت و حضور کم در فصل، هم امتیاز و هم قضاوت دربارهٔ «ارزش» را تحت تأثیر می‌گذارد.
- هر بازیکن ممکن است در چند فصل تکرار شده باشد (مستقل نبودن ردیف‌ها).

</div>

<div dir="rtl" style="text-align: right;">
<h3>8.2 تحلیل تکمیلی دوم</h3>
</div>

<div dir="rtl" style="text-align: right;">
<h3>8.3 تحلیل تکمیلی سوم</h3>
</div>

<div dir="rtl" style="text-align: right;">
<h3>8.4 تحلیل تکمیلی چهارم</h3>
</div>

<div dir="rtl" style="text-align: right;">
<h3>8.5 تحلیل تکمیلی پنجم</h3>
</div>